# HotpotQA · NARS + 8B + LoRA and Llama-3.1-8B (Colab, A100)

The only notebook that needs a GPU. One vLLM server holds Llama-3.1-8B-Instruct and the distilled adapter, so the
two systems share the same weights:
- **NARS + 8B + LoRA**: the adapter writes triples, aliases, plan and rules; ONA derives the answer.
- **Llama-3.1-8B**: the same 8B answers directly (the reference every other system is compared with).

**Before you run:** Runtime → **A100**; Colab secrets (🔑, *Notebook access* on): `DEEPINFRA_API_KEY` (the judge)
and `HF_TOKEN` (Hugging Face **read** token; the adapter is private). Then **Run all**. Setup about 10 minutes; then one cell per k, each ending with a comparison.

Answers go to Google Drive (`MyDrive/nars_lora/HotpotQA/results/`) as they finish; running again resumes. When it is
done, copy that folder's `answers/` and `traces/` into this repository's `HotpotQA/results/` (the baselines report then
includes these rows). The last cell releases the GPU.

In [ ]:
%%capture
!git clone -q https://github.com/MinaGabriel/NeuNARS.git /content/NeuNARS || git -C /content/NeuNARS pull -q
!git clone -q https://github.com/opennars/OpenNARS-for-Applications.git /content/ONA || true
!cd /content/ONA && git checkout -q dc4efd0abd520cdb79bf53bfa3c285ebb24f2e8a && sh build.sh
!pip install -q vllm prettytable

In [ ]:
import os, subprocess, sys, time, urllib.request
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

DATASET = "HotpotQA"
N_QUESTIONS = 200             # the first N questions; 5 for a quick check
WORKERS = 16
LORA_NAME = "NARS + 8B + LoRA v3"   # the row name; one per adapter, so answers never mix
ADAPTER_REVISION = "2685e7f2214b97aa0578ab9950891c76cdb61ebc"   # v3 (+ k=4 and k=8, DeepSeek-V3 and Llama-3.3-70B teachers)
# v2 = "86e4c6d5f9abb2e3b06615bb8d0f01d2b6c8e76a" (+ k=4, DeepSeek-V3) · v1 = "de96b5e36c845464201dad4d1c6b6648b523af70"
# to evaluate an earlier adapter, set both LORA_NAME and ADAPTER_REVISION to that version

os.chdir(f"/content/NeuNARS/{DATASET}")
sys.path.insert(0, "/content/NeuNARS/common")
os.environ["DEEPINFRA_API_KEY"] = userdata.get("DEEPINFRA_API_KEY")
assert os.path.exists("/content/ONA/NAR"), "ONA did not build: run the first cell without %%capture to see why"
drive.mount("/content/drive")
RESULTS = f"/content/drive/MyDrive/nars_lora/{DATASET}/results"

import experiment as ex
revision = ADAPTER_REVISION or HfApi().model_info(ex.ADAPTER_REPO, token=userdata.get("HF_TOKEN")).sha
adapter = snapshot_download(ex.ADAPTER_REPO, revision=revision, token=userdata.get("HF_TOKEN"), local_dir="/content/adapter")
print(f"adapter {ex.ADAPTER_REPO} @ {revision[:8]}")

server = subprocess.Popen(
    ["vllm", "serve", "unsloth/Meta-Llama-3.1-8B-Instruct", "--served-model-name", "llama-3.1-8b-instruct",
     "--dtype", "bfloat16", "--enable-lora", "--lora-modules", f"nars-lora={adapter}", "--max-lora-rank", "16",
     "--max-model-len", "16384", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for minute in range(20):      # the first start downloads 16 GB of weights
    time.sleep(30)
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print("vLLM is up")
        break
    except Exception:
        if server.poll() is not None:
            print(open("/content/vllm.log").read()[-3000:])
            raise SystemExit("vLLM stopped: see the log above")
        print(f"starting vLLM... {(minute + 1) / 2:.1f} min")

## Systems

In [ ]:
from openai import OpenAI

questions, question_hash = ex.load_questions("questions.pkl", N_QUESTIONS)
vllm = OpenAI(api_key="none", base_url="http://localhost:8000/v1", max_retries=2, timeout=600)
ex.check_models([(vllm, "nars-lora"), (vllm, "llama-3.1-8b-instruct"), (ex.deepinfra(), ex.JUDGE_MODEL)])
systems = {"Llama-3.1-8B": (vllm, "llama-3.1-8b-instruct", None),
           "Llama-3.1-8B + CoT": (vllm, "llama-3.1-8b-instruct", ex.COT),   # the same 8B, reasoning step by step
           LORA_NAME: (vllm, "nars-lora", "/content/ONA/NAR")}
ex.SIZES.setdefault(LORA_NAME, "8B")
ex.NARS_SYSTEMS.add(LORA_NAME)
COMPARE = [ex.REFERENCE, f"{ex.REFERENCE} + CoT", "NARS + 8B + LoRA", LORA_NAME]   # the 8B, the first adapter (if saved) and this one
print(f"{LORA_NAME} = adapter {ex.ADAPTER_REPO} @ {revision[:8]} · k = {', '.join(map(str, ex.K_VALUES[DATASET]))}")

def run_k(k):
    """Runs every missing answer at this k (a progress bar with money and running accuracy), then compares the systems."""
    ex.run_all(questions, question_hash, RESULTS, systems, [k], workers=WORKERS)
    print()
    ex.compare_k(RESULTS, COMPARE, k, N_QUESTIONS)

## k = 0

In [ ]:
run_k(0)

## k = 4

In [ ]:
run_k(4)

## k = 8

In [ ]:
run_k(8)

## Final report

Every system saved in this results folder (the 8B, the first adapter and this one), all k: accuracy, robustness
vs the 8B with 95% CIs, no-answer rates and cost. Saved as `report.txt` and `accuracy_vs_k.png`.

In [ ]:
ex.report(RESULTS, DATASET, N_QUESTIONS)

## Done: release the GPU

In [ ]:
from google.colab import runtime
server.terminate()
drive.flush_and_unmount()
runtime.unassign()